# Lesson 2.2 — Target Selection: Which Fruit, Which Pose?

Selection is **filter (ripe ∧ reachable) → rank (nearest) → commit**. Feasibility precedes cost; an empty feasible set yields `None`. (Greenhouse arm: L1=0.4, L2=0.3, reach ≈ 0.7 m.)

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, REACH_MAX)
checks = []
# hand-built world (all within/around the 0.7 m workspace):
#   F0 unripe & nearest, F1 ripe but out of reach, F2/F3/F4 ripe + reachable
world = GreenhouseWorld(fruit=[
    Fruit('F0', 0.18, 0.08, ripe=False),
    Fruit('F1', REACH_MAX+0.25, 0.0, ripe=True),
    Fruit('F2', 0.45, 0.00, ripe=True),
    Fruit('F3', 0.60, 0.00, ripe=True),
    Fruit('F4', 0.55, 0.00, ripe=True)],
    q=np.array([0.6, 0.8]))
tool = np.array([0.0, 0.0])
dets = model_perception(world, rng=np.random.default_rng(0))
targets, current = understand(dets, world, tool_xy=tool)


In [2]:
print('committed target:', current['id'] if current else None)
feasible = [t['id'] for t in targets if t['ripe'] and t['reachable']]
print('feasible (ripe & reachable):', feasible)
# F0 excluded (unripe), F1 excluded (unreachable)
checks.append('F0' not in feasible and 'F1' not in feasible)
# nearest feasible to the origin is F2 (0.45)
checks.append(current is not None and current['id'] == 'F2')
checks.append(current['ripe'] and current['reachable'])


committed target: F2
feasible (ripe & reachable): ['F2', 'F3']


### Feasibility precedes cost
F0 is *nearest* (≈ 0.20) but unripe, so it is never chosen — the filter dominates the distance rank.

In [3]:
f0 = next(t for t in targets if t['id']=='F0')
print('F0 dist=%.2f ripe=%s reachable=%s -> not selected' % (f0['dist'], f0['ripe'], f0['reachable']))
checks.append(f0['dist'] < current['dist'] and not f0['ripe'])  # nearer but excluded


F0 dist=0.20 ripe=False reachable=True -> not selected


In [4]:
# graceful empty case: make everything unripe -> current_target is None
for f in world.fruit: f.ripe = False
dets2 = model_perception(world, rng=np.random.default_rng(0))
_, current2 = understand(dets2, world, tool_xy=tool)
print('all-unripe -> committed target:', current2)
checks.append(current2 is None)
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


all-unripe -> committed target: None
5/5 checks passed.
All checks passed.
